In [2]:
import pandas as pd
import os
import sys
from glob import glob
import librosa


In [3]:

# Load CSVs and create DataFrame with file paths
shallow_train_df_041326 = pd.read_pickle("/media/auk/projects/gak76/vira_beg_outputs/training_data/splits/shallow_train_df_gk_40926.pkl")
shallow_test_df_041326  = pd.read_pickle("/media/auk/projects/gak76/vira_beg_outputs/training_data/splits/shallow_test_df_gk_40926.pkl")



print("Train shape:", shallow_train_df_041326.shape)
print("Test shape:", shallow_test_df_041326.shape)
print("Train index names:", shallow_train_df_041326.index.names)
print("Test index names:", shallow_train_df_041326.index.names)



Train shape: (3172, 1)
Test shape: (377, 1)
Train index names: ['file', 'start_time', 'end_time']
Test index names: ['file', 'start_time', 'end_time']


In [ ]:
# ---------------- TRAIN ----------------
print(f"\nLoaded {len(shallow_train_df_041326)} TRAIN clips")
print(f"Sample file path: {shallow_train_df_041326.index[0][0]}")

print("Verifying TRAIN clip files...")
invalid_clips = []

unique_files = shallow_train_df_041326.index.get_level_values("file").unique()


for i, file_path in enumerate(unique_files):
    if i % 50 == 0:
        print(f"Checked {i}/{len(unique_files)} files")

    if not os.path.exists(file_path):
        invalid_clips.append((file_path, "File not found"))
        continue
    
    try:
        y, sr = librosa.load(file_path, sr=None)
        if y.shape[0] == 0:
            invalid_clips.append((file_path, "Zero samples"))
    except Exception as e:
        invalid_clips.append((file_path, str(e)))

if invalid_clips:
    print(f"CRITICAL: Issues found in TRAIN clips")
    for file_path, reason in invalid_clips:
        print(f"  - {file_path}: {reason}")
    sys.exit(1)
else:
    print("TRAIN files verified successfully!")


Loaded 3172 TRAIN clips
Sample file path: /home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/104728661.wav
Verifying TRAIN clip files...
Checked 0/1652 files


In [ ]:
# ---------------- TEST ----------------
print(f"\nLoaded {len(shallow_test_df_041326)} TEST clips")
print(f"Sample file path: {shallow_test_df_041326.index[0][0]}")

print("Verifying TEST clip files...")
invalid_clips = []



unique_files = shallow_test_df_041326.index.get_level_values("file").unique()

for file_path in unique_files:
    
    if not os.path.exists(file_path):
        invalid_clips.append((file_path, "File not found"))
        continue
    
    try:
        y, sr = librosa.load(file_path, sr=None)
        if y.shape[0] == 0:
            invalid_clips.append((file_path, "Zero samples"))
    except Exception as e:
        invalid_clips.append((file_path, str(e)))

if invalid_clips:
    print(f"CRITICAL: Issues found in TEST clips")
    for file_path, reason in invalid_clips:
        print(f"  - {file_path}: {reason}")
    sys.exit(1)
else:
    print("TEST files verified successfully!")

In [4]:
# Run this FIRST, before importing bioacoustics_model_zoo
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("GPUs seen by TensorFlow:", gpus)

if gpus:
    try:
        # choose one GPU; change index if needed
        tf.config.set_visible_devices(gpus[1], "GPU")

        # cap memory usage on that GPU
        tf.config.set_logical_device_configuration(
            gpus[1],
            [tf.config.LogicalDeviceConfiguration(memory_limit=5000)]
        )

        logical_gpus = tf.config.list_logical_devices("GPU")
        print("Logical GPUs:", logical_gpus)

    except RuntimeError as e:
        print("TensorFlow GPU setup error:", e)

2026-04-13 11:54:15.293506: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations 

GPUs seen by TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Logical GPUs: [LogicalDevice(name='/device:GPU:0', device_type='GPU')]


I0000 00:00:1776095659.452845 3303252 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5000 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:21:00.0, compute capability: 8.6


In [5]:
import bioacoustics_model_zoo as bmz

model = bmz.Perch2()

/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/tensorflow_hub/__init__.py:61: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT
                    

In [7]:
# ---------------- EMBEDDINGS ----------------
# TRAIN embeddings
# ---------------- EMBEDDINGS ----------------

train_embeddings = model.embed(shallow_train_df_041326, batch_size=1, num_workers=0)

# FIX: make directory
output_dir = "/media/auk/projects/gak76/vira_beg_outputs/embeddings"
os.makedirs(output_dir, exist_ok=True)

train_output = f"{output_dir}/shallow_train_embeddings_041326.pkl"
train_embeddings.to_pickle(train_output)

print(f"Saved TRAIN embeddings: {len(train_embeddings)} rows to {train_output}")

  0%|          | 0/3172 [00:00<?, ?it/s]

/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/audio.py:1757: UserWarning: Audio object is shorter than requested duration: 0.706625 sec instead of 1.0 sec
  warnings.warn(error_msg)
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/audio.py:1757: UserWarning: Audio object is shorter than requested duration: 0.39740625 sec instead of 1.0 sec
  warnings.warn(error_msg)
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/audio.py:1757: UserWarning: Audio object is shorter than requested duration: 0.7845625 sec instead of 1.0 sec
  warnings.warn(error_msg)
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/audio.py:1757: UserWarning: Audio object is shorter than requested duration: 0.786 sec instead of 1.0 sec
  warnings.warn(error_msg)
/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/audio.py:1757: UserWarning: Audio object is shorter than

Saved TRAIN embeddings: 3172 rows to /media/auk/projects/gak76/vira_beg_outputs/embeddings/shallow_train_embeddings_041326.pkl


In [8]:
# TEST embeddings
test_embeddings = model.embed(shallow_test_df_041326, batch_size=1, num_workers=0)

# make sure directory exists
output_dir = "/media/auk/projects/gak76/vira_beg_outputs/embeddings"
os.makedirs(output_dir, exist_ok=True)

test_output = f"{output_dir}/shallow_test_embeddings_041326.pkl"
test_embeddings.to_pickle(test_output)

print(f"Saved TEST embeddings: {len(test_embeddings)} rows to {test_output}")

  0%|          | 0/377 [00:00<?, ?it/s]

/home/sml161/miniconda3/envs/perch2/lib/python3.11/site-packages/opensoundscape/audio.py:1757: UserWarning: Audio object is shorter than requested duration: 0.89703125 sec instead of 1.0 sec
  warnings.warn(error_msg)


Saved TEST embeddings: 377 rows to /media/auk/projects/gak76/vira_beg_outputs/embeddings/shallow_test_embeddings_041326.pkl
